# Module 5: Enterprise Integration with MCP and Secure Adapters

This notebook implements a mock **ERP_Connector MCP Server** with safe tools and RBAC controls for SnapAudit.


## 1. Setup and Goals

We want an agent that can read/write to a ledger safely:
- `approve_transaction`
- `flag_fraud`
- RBAC for approvals over $10k (requires `finance_admin` + digital key)


In [ ]:
from pathlib import Path
import sys
import json

module_dir = Path.cwd()
if str(module_dir) not in sys.path:
    sys.path.append(str(module_dir))

print("✓ Setup complete")


## 2. Load MCP Adapter Components

Import the secure server, ledger, and client abstractions.


In [ ]:
from mcp_secure_adapter import (
    MockERPLedger,
    ERPConnectorMCPServer,
    SnapAuditAgentClient,
    RBACPolicy,
)

print("✓ MCP components imported")


## 3. Initialize Ledger and Server

Start with a mock ERP ledger and MCP server.


In [ ]:
ledger = MockERPLedger()
mcp_server = ERPConnectorMCPServer(ledger)
agent = SnapAuditAgentClient(mcp_server)

print("✓ ERP MCP server initialized")
print(f"Transactions loaded: {len(ledger.transactions)}")


### Inspect Seed Transactions


In [ ]:
ledger.list_transactions()


## 4. Tool-by-Tool MCP Testing

Validate each exposed tool directly.


In [ ]:
    # Read tool
n = mcp_server.get_transaction("T1002")
print(json.dumps(n, indent=2))


In [ ]:
    # Flag tool
flag_result = mcp_server.flag_fraud(
    transaction_id="T1002",
    actor_role="manager",
    reason="Potential duplicate receipt",
)
print(json.dumps(flag_result, indent=2))


In [ ]:
    # Approve tool under threshold
approve_small = mcp_server.approve_transaction(
    transaction_id="T1001",
    actor_role="agent",
)
print(json.dumps(approve_small, indent=2))


## 5. RBAC Scenarios for High-Value Approvals

For amounts over $10,000:
- `agent` should be denied
- `finance_admin` without key should be denied
- `finance_admin` with correct key should be approved


In [ ]:
    # Case A: agent denied over threshold
case_a = mcp_server.approve_transaction("T1003", actor_role="agent")
print("Case A (agent):", case_a.get("ok"), case_a.get("error"))

# Case B: finance_admin without key denied
case_b = mcp_server.approve_transaction("T1003", actor_role="finance_admin")
print("Case B (admin no key):", case_b.get("ok"), case_b.get("error"))

# Case C: finance_admin with key approved
case_c = mcp_server.approve_transaction(
    "T1003",
    actor_role="finance_admin",
    digital_key=RBACPolicy.HIGH_VALUE_KEY,
)
print("Case C (admin + key):", case_c.get("ok"), case_c.get("message"))


## 6. Agent-Driven End-to-End Flow

Simulate SnapAudit policy output and let the client invoke MCP tools accordingly.


In [ ]:
    # Policy says compliant -> try approval
flow_1 = agent.process_transaction(
    transaction_id="T1004",
    policy_approved=True,
    actor_role="manager",
)
print("Flow 1 status:", flow_1.get("ok"), flow_1.get("transaction", {}).get("status"), flow_1.get("error"))

# Policy says non-compliant -> flag
flow_2 = agent.process_transaction(
    transaction_id="T1002",
    policy_approved=False,
    actor_role="agent",
)
print("Flow 2 status:", flow_2.get("ok"), flow_2.get("transaction", {}).get("status"), flow_2.get("error"))


## 7. Audit Log Review

Every tool call is logged for compliance tracing.


In [ ]:
for row in mcp_server.audit_log:
    print(row)


## 8. Assertions (Smoke Test)


In [ ]:
assert case_a["ok"] is False
assert case_b["ok"] is False
assert case_c["ok"] is True
assert flow_2["ok"] is True and flow_2["transaction"]["status"] == "flagged"

print("✓ Module 5 MCP smoke checks passed")
